# ConvNeXt Small + Query2Label — Nutrition5k, 100 clases

Notebook basado en `COLAB_QUERY2LABEL.md`. Arquitectura: consultas de atención, MLP compartido 768 → 384 → 192, ASL/BCE y ajuste de la última etapa del encoder después de 3 épocas de warmup.

La pérdida de masa total se divide por la media de **train**, calculada al preparar la corrida (≈219,11 g con los datos actuales); la pérdida de gramos de ingredientes conserva 300 g. No fijar 219,11 manualmente.

Las celdas de entrenamiento están desactivadas por defecto. Primero ejecutar preparación y verificación; después habilitar explícitamente la corrida deseada. Evaluación e inferencia también son opcionales. Elegir **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU** antes de comenzar.


## Antes de Colab: archivos en Git y Drive

**Git:** subir `src/`, `scripts/`, `config/`, `tests/`, `requirements.txt`, `pyproject.toml`, `.gitignore`, los documentos del proyecto y este notebook. Conservar la estructura interna de IngredientMassEstimation. El repositorio puede contener esta carpeta o publicarla directamente en su raíz. No subir datos, entornos, cachés, pesos ni resultados.

**Drive:** crear `MyDrive/IngredientMassEstimation/` y subir `nutrition5k_rgb.zip`. El ZIP debe contener `nutrition5k_dataset/metadata`, `nutrition5k_dataset/dish_ids` y `nutrition5k_dataset/imagery/realsense_overhead`.

En PowerShell, desde la carpeta local del proyecto (ejecutar manualmente; reutilizar un ZIP existente):

```powershell
Compress-Archive -LiteralPath ".\data\raw\nutrition5k_dataset" -DestinationPath ".\data\nutrition5k_rgb.zip"
```

Para continuar una corrida local, subir **todo el contenido** de `artifacts/query2label_100_asl/` a `MyDrive/IngredientMassEstimation/runs/query2label_100_asl/`: `last.pt`, `best.pt`, `history.csv`, `config.json` y `latest_validation.json`. Así se conserva también el mejor modelo previo aunque las siguientes épocas no mejoren. No hace falta copiar `data/processed_100`: se regenera con los mismos datos, seed y configuración.


## Opcional: una época local en CPU

Estos comandos son una referencia para ejecutar **manualmente en PowerShell**, no una celda de Colab. Desde IngredientMassEstimation, usando el entorno disponible de OmniFood:

```powershell
$pythonExe = "..\OmniFood8K\Code\.venv\Scripts\python.exe"
& $pythonExe scripts/prepare_nutrition5k.py --config config/query2label_100.json --output-dir data/processed_100
& $pythonExe scripts/train_image_model.py --check-data
& $pythonExe -u scripts/train_image_model.py --config config/query2label_100.json --device cpu --workers 0 --epochs 1 --output-dir artifacts/query2label_100_asl
```

La tercera línea entrena una época completa y puede tardar mucho en CPU; requiere descargar inicialmente los pesos ImageNet. Mantener batch 8, acumulación 4 y el resto de la configuración para continuar en Colab. AMP se desactiva automáticamente en CPU, sin cambiar `amp: true` en el config. Si la carpeta de salida ya contiene una corrida, usar otra carpeta o reanudar conscientemente.

Después subir la carpeta de la corrida a Drive y usar modo `resume` abajo. `TOTAL_EPOCHS = 30` continúa desde la época 2 hasta un máximo total de 30 (sujeto a early stopping); `2` permite probar solamente la siguiente época. Cambiar dispositivo y workers no da identidad bit a bit.


## 1. GPU y Drive

In [ ]:
import torch
assert torch.cuda.is_available(), "Este runtime no tiene GPU CUDA. Cambiá el entorno de ejecución."
print(torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount("/content/drive")

## 2. Obtener el código

Reemplazar `URL` por el repositorio publicado. Para un repo privado, agregar `GITHUB_TOKEN` a los secretos de Colab y habilitar acceso al notebook. Para un repo público, elegir `PRIVATE_REPO = False`. No pegar credenciales en el código.


In [ ]:
from pathlib import Path
import base64, os, subprocess, sys

REPO = Path("/content/nutrition-code")
URL = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
PRIVATE_REPO = True

assert "TU_USUARIO" not in URL and "TU_REPOSITORIO" not in URL, "Completá URL con tu repositorio."
environment = os.environ.copy()
if PRIVATE_REPO:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    assert token, "Configurá el secreto GITHUB_TOKEN."
    authorization = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    environment.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader",
                        "GIT_CONFIG_VALUE_0": f"Authorization: Basic {authorization}"})
try:
    command = (["git", "-C", str(REPO), "pull", "--ff-only"] if (REPO / ".git").exists()
               else ["git", "clone", URL, str(REPO)])
    result = subprocess.run(command, env=environment, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Git falló. Verificá URL, permisos del token y cambios locales; no se imprime la credencial.")
finally:
    del environment
    if PRIVATE_REPO:
        del token, authorization

candidates = [REPO / "IngredientMassEstimation", REPO]
PROJECT = next((p for p in candidates if (p / "config/query2label_100.json").is_file()), None)
assert PROJECT is not None, "El repositorio no contiene la nueva variante."
os.chdir(PROJECT)
print("Project:", PROJECT)

## 3. Dependencias mínimas

Usar PyTorch/torchvision con CUDA del runtime. No copiar el entorno Windows ni instalar los paquetes de OmniFood.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "Pillow>=10,<13", "tqdm>=4.66,<5"], check=True)
import torch, torchvision, numpy
assert hasattr(torch.amp, "GradScaler"), "Hace falta PyTorch >= 2.3."
assert torch.cuda.is_available()
print("Torch:", torch.__version__, "Torchvision:", torchvision.__version__, "NumPy:", numpy.__version__)

## 4. Copiar y extraer los datos en /content

El dataset se lee desde el almacenamiento local de Colab. Drive conserva ZIP, checkpoints y resultados entre sesiones.

In [ ]:
import shutil, zipfile

DRIVE = Path("/content/drive/MyDrive/IngredientMassEstimation")
DATA = Path("/content/nutrition5k/nutrition5k_dataset")
if not (DATA / "metadata").is_dir():
    archive = Path("/content/nutrition5k_rgb.zip")
    source_archive = DRIVE / "nutrition5k_rgb.zip"
    assert source_archive.is_file(), f"Falta subir: {source_archive}"
    shutil.copy2(source_archive, archive)
    destination = DATA.parent
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as source:
        for name in source.namelist():
            resolved = (destination / name).resolve()
            if not resolved.is_relative_to(destination.resolve()):
                raise ValueError("Ruta insegura en el ZIP")
        source.extractall(destination)
for folder in ("metadata", "dish_ids", "imagery/realsense_overhead"):
    assert (DATA / folder).is_dir(), f"Revisá la estructura interna del ZIP: falta {folder}"
print("Dataset:", DATA)

## 5. Preparar etiquetas y verificar sin entrenar

Esperado: 100 clases, 2332 train, 423 validación y 507 test. La comprobación calcula la escala de masa de train, sin cargar el modelo ni descargar pesos.

In [ ]:
import json

CONFIG = "config/query2label_100.json"
PROCESSED = "data/processed_100"
subprocess.run([sys.executable, "scripts/prepare_nutrition5k.py",
                "--config", CONFIG, "--dataset-root", str(DATA),
                "--output-dir", PROCESSED], check=True)
checked = subprocess.run([sys.executable, "scripts/train_image_model.py", "--check-data",
                          "--config", CONFIG, "--processed-dir", PROCESSED,
                          "--dataset-root", str(DATA)], check=True, capture_output=True, text=True)
print(checked.stdout)
verification = json.loads(checked.stdout)
assert verification["classes"] == 100
assert verification["splits"] == {"train": 2332, "validation": 423, "test": 507}, "El dataset difiere del local."
assert verification["config"]["mass_normalization"] == "train_mean"
assert verification["config"]["ingredient_mass_scale"] == 300.0
print("Escala de masa total (train):", verification["config"]["mass_scale"], "g")

## 6. Entrenar ASL o continuar desde CPU/Colab

Elegir `MODE = "new"` para comenzar de cero en una carpeta vacía, o `"resume"` para continuar desde `last.pt` subido a Drive. Definir primero `TOTAL_EPOCHS` y luego habilitar `RUN_TRAINING` cuando quieras entrenar. El directorio ya usado no se sobrescribe con una corrida nueva.

Los pesos iniciales ImageNet (~192 MB) se guardan en `torch_cache`. Cada cierre de época guarda `best.pt`, `last.pt` y el historial en Drive. Una interrupción a mitad de época obliga a repetirla. Conservar la misma configuración al reanudar: solo puede cambiar el límite total de épocas. No mezclar ASL y BCE.


In [ ]:
RUN_TRAINING = False  # Cambiar a True solamente cuando quieras ejecutar entrenamiento.
MODE = "resume"        # "new" para empezar desde cero; "resume" para continuar last.pt.
TOTAL_EPOCHS = 30      # Límite TOTAL, no cantidad de épocas adicionales.
RUN = DRIVE / "runs/query2label_100_asl"

arguments = [sys.executable, "-u", "scripts/train_image_model.py",
             "--config", CONFIG, "--processed-dir", PROCESSED,
             "--dataset-root", str(DATA), "--output-dir", str(RUN),
             "--torch-cache", str(DRIVE / "torch_cache"),
             "--device", "cuda", "--workers", "2", "--epochs", str(TOTAL_EPOCHS)]

if RUN_TRAINING:
    assert MODE in ("new", "resume"), "MODE debe ser new o resume."
    training_arguments = arguments.copy()
    if MODE == "resume":
        assert (RUN / "last.pt").is_file(), "Subí la corrida local o elegí MODE = new."
        assert (RUN / "best.pt").is_file(), "Subí también best.pt para conservar el mejor modelo previo."
        training_arguments += ["--resume", str(RUN / "last.pt")]
    else:
        assert not RUN.exists() or not any(RUN.iterdir()), "Usá otra carpeta para una corrida nueva."
    subprocess.run(training_arguments, check=True)
else:
    print("Entrenamiento desactivado. Revisá MODE, TOTAL_EPOCHS y RUN antes de habilitarlo.")

### Revisar el progreso

El ajuste del encoder empieza en época 4. Observar tiempo por época y unidades de Colab antes de completar 30 épocas. Para una **corrida nueva** con menos memoria, usar un config con batch 4 y acumulación 8; no cambiar esos valores al reanudar una corrida existente.


In [ ]:
import csv

history = RUN / "history.csv"
if history.is_file():
    with history.open(newline="", encoding="utf-8") as source:
        rows = list(csv.DictReader(source))
    for row in rows[-5:]:
        print(row)
    if rows:
        print("Última época:", rows[-1]["epoch"], "| segundos:", rows[-1]["seconds"])
else:
    print("Todavía no hay historial en", RUN)

## 7. Opcional: comparar con BCE

Corrida separada, con los mismos datos, seed y arquitectura. Habilitar solo si el presupuesto lo permite. Seleccionar ASL/BCE con validación. Si ya hay una corrida BCE, elegir `BCE_MODE = "resume"`.


In [ ]:
RUN_BCE_TRAINING = False
BCE_MODE = "new"
RUN_BCE = DRIVE / "runs/query2label_100_bce"
bce_arguments = [sys.executable, "-u", "scripts/train_image_model.py",
                 "--config", CONFIG, "--processed-dir", PROCESSED,
                 "--dataset-root", str(DATA), "--output-dir", str(RUN_BCE),
                 "--torch-cache", str(DRIVE / "torch_cache"), "--device", "cuda",
                 "--workers", "2", "--presence-loss", "bce", "--epochs", str(TOTAL_EPOCHS)]
if RUN_BCE_TRAINING:
    assert BCE_MODE in ("new", "resume")
    if BCE_MODE == "resume":
        assert (RUN_BCE / "last.pt").is_file() and (RUN_BCE / "best.pt").is_file()
        bce_arguments += ["--resume", str(RUN_BCE / "last.pt")]
    else:
        assert not RUN_BCE.exists() or not any(RUN_BCE.iterdir()), "Usá otra carpeta o reanudá."
    subprocess.run(bce_arguments, check=True)
else:
    print("Comparación BCE desactivada.")

## 8. Calibrar con validación y evaluar test

Ejecutar después de elegir la variante con validación. Estas rutas corresponden a ASL. El umbral calibrado cambia los reportes de presencia; no recorta las proporciones ni cambia los nutrientes.

In [ ]:
RUN_CALIBRATION = False
VALIDATION = DRIVE / "reports/query2label_asl_validation"
CHECKPOINT = VALIDATION / "calibrated.pt"
if RUN_CALIBRATION:
    assert (RUN / "best.pt").is_file(), "Primero necesitás un modelo entrenado."
    subprocess.run([sys.executable, "scripts/evaluate_image_model.py",
                    "--checkpoint", str(RUN / "best.pt"), "--dataset-root", str(DATA),
                    "--processed-dir", PROCESSED, "--split", "validation", "--calibrate",
                    "--output-dir", str(VALIDATION), "--device", "cuda"], check=True)
else:
    print("Calibración desactivada.")

In [ ]:
RUN_TEST_EVALUATION = False
if RUN_TEST_EVALUATION:
    assert CHECKPOINT.is_file(), "Primero calibrá con validación."
    subprocess.run([sys.executable, "scripts/evaluate_image_model.py",
                    "--checkpoint", str(CHECKPOINT), "--dataset-root", str(DATA),
                    "--processed-dir", PROCESSED, "--split", "test",
                    "--output-dir", str(DRIVE / "reports/query2label_asl_test"),
                    "--device", "cuda"], check=True)
else:
    print("Evaluación de test desactivada.")

## 9. Foto del celular e intermedios

Subir una foto a `MyDrive/IngredientMassEstimation/photos/plato.png`. Se exportan JSON, crop, tensores y mapas de atención. La atención no es segmentación ni prueba de presencia; una foto fuera de Nutrition5k puede tener errores por cambio de dominio.

In [ ]:
RUN_INFERENCE = False
PHOTO = DRIVE / "photos/plato.png"
if RUN_INFERENCE:
    assert CHECKPOINT.is_file(), "Falta el checkpoint calibrado."
    assert PHOTO.is_file(), f"Subí una foto a {PHOTO}"
    subprocess.run([sys.executable, "scripts/infer_image_model.py",
                    "--checkpoint", str(CHECKPOINT), "--image", str(PHOTO),
                    "--output-dir", str(DRIVE / "predictions"), "--device", "cuda",
                    "--save-attention", "--top-k", "10"], check=True)
else:
    print("Inferencia desactivada.")

## Archivos persistentes

- `runs/query2label_100_asl/`: checkpoints completos, config, historial y validación más reciente.
- `torch_cache/`: pesos ImageNet descargados al comenzar una corrida nueva.
- `reports/query2label_asl_validation/`: métricas y `calibrated.pt`.
- `reports/query2label_asl_test/`: métricas y predicciones de test.
- `predictions/plato/`: predicción e intermedios de la foto.

Para inferencia local alcanza con descargar `best.pt` o `calibrated.pt` y usar los scripts con `--device cpu`. Para reanudar entrenamiento conservar `last.pt` y toda la carpeta de corrida. No se necesitan Depth Anything, pesos OmniFood ni extracción de vectores.
